# H_TextMention — 설명의 패턴 언급 여부

Purchase-level Recall within product-group cells; C2−C4. Reuses the shared A/main predictions. The original analysis code survived.

**탐색적 분석.** Uploaded notebook cells were reorganized after the experiment. Saved CSV/JSON/bootstrap arrays are unchanged. Outputs and execution counts have been cleared. The core is pinned to the recorded experiment commit; dependency versions may still differ.


In [ ]:
HYPOTHESIS = "H_TextMention"
# Fresh Colab runtime: run this setup once per notebook.
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/msrs_paper (1)"  # Change for your shortcut.
drive.mount("/content/drive")

CORE_COMMIT = "e6b73dc4a8aee549b4310cf4f3266e373b8632ca"
CORE_DIR = Path("/content/msrs_kwangsuk_core")
REPO_URL = "https://github.com/3sunghynix/multimodal-recsys.git"
if any(name.startswith("msrs_paper") for name in sys.modules):
    raise RuntimeError("Restart the Colab session before running setup again.")
if not CORE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CORE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CORE_DIR), "checkout", "--detach", CORE_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(CORE_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != CORE_COMMIT:
    raise RuntimeError("The existing core checkout has a different commit; use a fresh runtime.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(CORE_DIR / "paper/requirements.txt")
], check=True)
sys.path.insert(0, str(CORE_DIR / "paper"))
from msrs_paper.config import load_all, Paths
from msrs_paper import metrics
from msrs_paper.utils import verify_manifest
cfg, exps, plan = load_all()
paths = Paths(DRIVE_ROOT)  # Do not create a missing shared-data folder.
assert (paths.root / "manifest.json").is_file(), "Check DRIVE_ROOT and the shared shortcut."
manifest = verify_manifest(paths)
print("Verified files:", len(manifest["files"]))
print("Core commit:", actual_commit)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_OUTPUT_ROOT = Path("/content/drive/MyDrive/kwangsuk_reproductions") / stamp / HYPOTHESIS

def historical_folder(suffix, timestamp):
    candidates = [
        p / timestamp for p in Path("/content/drive/MyDrive").iterdir()
        if p.is_dir() and p.name.endswith(suffix) and (p / timestamp).is_dir()
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"Set the historical input folder explicitly: {candidates}")
    return candidates[0]


## 1. 비교 그룹
Both description groups require at least 300 purchases in a product group.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from msrs_paper import metrics

root = Path(DRIVE_ROOT)
items = pd.read_parquet(root / "processed/items.parquet")
targets = pd.read_parquet(root / "processed/main/test_targets.parquet")

# Match product information to each test purchase.
assert items["idx"].is_unique, "Duplicate product indices found."

flag_map = items.set_index("idx")["desc_mentions_pattern"]
description_flag = targets["item_idx"].map(flag_map)

assert description_flag.notna().all(), "Some products lack a description flag."
assert description_flag.isin([True, False]).all()

# Keep patterned products purchased for the first time by that customer.
mask = (
    targets["appearance"].eq("pattern")
    & ~targets["is_repurchase"]
    & targets["pgroup"].notna()
)

test = targets.loc[mask].copy()
test["description_group"] = np.where(
    description_flag.loc[mask].astype(bool),
    "mentioned",
    "not_mentioned"
)

counts = pd.crosstab(
    test["pgroup"], test["description_group"]
).reindex(columns=["not_mentioned", "mentioned"], fill_value=0)

# Fix eligible categories using purchase counts, not model results.
MIN_PURCHASES = 300
eligible = counts.index[
    (counts["not_mentioned"] >= MIN_PURCHASES)
    & (counts["mentioned"] >= MIN_PURCHASES)
].tolist()

print("Test-purchase counts:")
print(counts.to_string())
print("\nEligible categories:", eligible)

assert eligible, "No categories have enough purchases in both groups."

test = test[test["pgroup"].isin(eligible)].reset_index(drop=True)

# Each category's weight is its share of eligible purchases.
weights = counts.loc[eligible].sum(axis=1).astype(float)
weights = weights / weights.sum()

print("\nFixed category weights:")
print(weights.to_string())
print("\nReady: comparison groups have been defined.")

## 2. 재채점과 고객 부트스트랩
New outputs go to a fresh personal reproduction folder.

In [ ]:
from datetime import datetime, timezone
import json
import subprocess

SEEDS = [42, 43, 44]
K = 300
N_BOOT = 1000
BOOT_SEED = 42

# Use one common customer ordering for all models and seeds.
all_users = np.sort(targets["user"].unique())
customer_rows = metrics.target_rows(all_users, test)
n_users = len(all_users)

# For each category:
# cell 0 = description not mentioned; cell 1 = mentioned.
category_index = {name: i for i, name in enumerate(eligible)}
cell_ids = (
    test["pgroup"].map(category_index).to_numpy(dtype=int) * 2
    + test["description_group"].eq("mentioned").to_numpy(dtype=int)
)

n_cells = len(eligible) * 2
flat_cells = customer_rows * n_cells + cell_ids

# Number of purchases for each customer and comparison cell.
purchase_counts = np.bincount(
    flat_cells, minlength=n_users * n_cells
).reshape(n_users, n_cells)

def load_customer_hits(condition, seed):
    folder = (
        root / "outputs" / "A" / "main"
        / f"{condition}_seed{seed}"
    )
    users = np.load(folder / "users.npy", allow_pickle=False)
    candidates = np.load(
        folder / "candidates.npy",
        mmap_mode="r",
        allow_pickle=False
    )

    assert users.ndim == 1
    assert len(np.unique(users)) == len(users)
    assert candidates.ndim == 2
    assert candidates.shape[0] == len(users)
    assert candidates.shape[1] >= K

    # Match each purchase to this particular file's customer ordering.
    prediction_rows = metrics.target_rows(users, test)
    hits = metrics.hits_at_k(
        candidates,
        prediction_rows,
        test["item_idx"].to_numpy(),
        K
    ).astype(np.int64)

    return np.bincount(
        flat_cells,
        weights=hits,
        minlength=n_users * n_cells
    ).reshape(n_users, n_cells)

real_by_seed = []
shuffled_by_seed = []

for seed in SEEDS:
    print(f"Reading predictions for seed {seed}...")
    real_by_seed.append(load_customer_hits("C2", seed))
    shuffled_by_seed.append(load_customer_hits("C4", seed))

# Average matched hits across the three trained seeds.
real_hits = np.mean(real_by_seed, axis=0)
shuffled_hits = np.mean(shuffled_by_seed, axis=0)

totals = purchase_counts.sum(axis=0)
real_recall = real_hits.sum(axis=0) / totals
shuffled_recall = shuffled_hits.sum(axis=0) / totals
contributions = (real_recall - shuffled_recall).reshape(-1, 2)

w = weights.loc[eligible].to_numpy()

# Positive = more image contribution when pattern is not mentioned.
category_differences = contributions[:, 0] - contributions[:, 1]
estimate = float(w @ category_differences)

# Paired customer bootstrap:
# the same sampled customers are used for every condition and group.
rng = np.random.default_rng(BOOT_SEED)
hit_difference = real_hits - shuffled_hits
bootstrap_estimates = []

for _ in range(N_BOOT):
    sampled = rng.integers(0, n_users, size=n_users)
    multiplicity = np.bincount(sampled, minlength=n_users)

    denominators = multiplicity @ purchase_counts
    if np.any(denominators == 0):
        continue

    boot_contribution = (
        (multiplicity @ hit_difference) / denominators
    ).reshape(-1, 2)

    bootstrap_estimates.append(
        float(w @ (boot_contribution[:, 0] - boot_contribution[:, 1]))
    )

assert len(bootstrap_estimates) >= 950, (
    "Too many bootstrap samples had empty groups."
)

low, high = np.quantile(bootstrap_estimates, [0.025, 0.975])

rows = []
for i, category in enumerate(eligible):
    for j, group in enumerate(["not_mentioned", "mentioned"]):
        cell = 2 * i + j
        rows.append({
            "category": category,
            "description_group": group,
            "purchases": int(totals[cell]),
            "real_recall_at_300": float(real_recall[cell]),
            "shuffled_recall_at_300": float(shuffled_recall[cell]),
            "image_contribution": float(contributions[i, j]),
        })

table = pd.DataFrame(rows)

seed_estimates = []
for real, shuffled in zip(real_by_seed, shuffled_by_seed):
    contribution = (
        (real - shuffled).sum(axis=0) / totals
    ).reshape(-1, 2)
    seed_estimates.append(
        float(w @ (contribution[:, 0] - contribution[:, 1]))
    )

summary = {
    "status": "exploratory; previously examined main test split",
    "contrast": "not_mentioned minus mentioned image contribution",
    "measure": "C2 minus C4; purchase-level Recall@300 within cells",
    "segment": "pattern products, non_repurchase",
    "eligible_categories": eligible,
    "category_weights": weights.to_dict(),
    "min_cell_purchases": MIN_PURCHASES,
    "seeds": SEEDS,
    "estimate": estimate,
    "ci95_customer_bootstrap": [float(low), float(high)],
    "seed_estimates": dict(zip(map(str, SEEDS), seed_estimates)),
    "bootstrap_samples": len(bootstrap_estimates),
    "bootstrap_seed": BOOT_SEED,
    "code_commit": subprocess.check_output(
        ["git", "-C", str(CORE_DIR), "rev-parse", "HEAD"],
        text=True
    ).strip(),
}

# Save in your own separate folder, not the team's results folders.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = RUN_OUTPUT_ROOT
out.mkdir(parents=True, exist_ok=False)

table.to_csv(out / "group_scores.csv", index=False)
counts.to_csv(out / "purchase_counts.csv")
np.save(out / "bootstrap_estimates.npy", bootstrap_estimates)

with open(out / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nGroup results:")
print(table.round(6).to_string(index=False))

print(f"\nDifference: {estimate:+.6f}")
print(f"95% customer-bootstrap interval: [{low:+.6f}, {high:+.6f}]")
print("Per-seed differences:", dict(zip(SEEDS, seed_estimates)))
print("\nSaved to:", out)